# RETRIEVAL - BM25 + TOP K

In [ ]:
!pip -q install rank-bm25

import json
import csv
import re
import time
from typing import Dict, Any, List, Tuple, Optional

import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from rank_bm25 import BM25Okapi


PQA_L_TEST_PATH = "/content/drive/MyDrive/BioBERT_PubMedQA_Checkpoints/ori_pqal.json"
PQA_U_PATH      = "/content/drive/MyDrive/BioBERT_PubMedQA_Checkpoints/ori_pqau.json"
BIOBERT_PATH    = "/content/drive/MyDrive/biobert/biobert_final_success"
OUTPUT_CSV_PATH = "/content/drive/MyDrive/lap/lap_lap/pqaL_test_biobert_bm25_top5.csv"

TOP_K = 5
MAX_BM25_SENTENCES: Optional[int] = 203225
PROGRESS_EVERY = 50000

BM25_QUERY_MODE = "question" 

BATCH_SIZE = 16          
MAX_LENGTH = 256

LIMIT_PQA_L_SAMPLES: Optional[int] = 500 

WRITE_LOG_EVERY = 10
TIMING_LOG_EVERY = 200


def normalize_ws(s: str) -> str:
    return re.sub(r"\s+", " ", str(s)).strip()

def flatten_contexts(contexts: List[str]) -> str:
    return normalize_ws(" ".join(normalize_ws(x) for x in contexts if normalize_ws(x)))

def bm25_tokenize(text: str) -> List[str]:
    text = text.lower()
    return re.findall(r"[a-z0-9]+", text)


def build_bm25_from_pqa_u_sentence_level(
    pqa_u: Dict[str, Any],
    max_sentences: Optional[int] = None,
    progress_every: int = 50000
) -> Tuple[BM25Okapi, List[str]]:
    t0 = time.time()
    corpus_sentences: List[str] = []

    print("Collecting sentences from PQA-U...")
    for _, item in pqa_u.items():
        for sent in item.get("CONTEXTS", []):
            sent = normalize_ws(sent)
            if not sent:
                continue

            corpus_sentences.append(sent)

            if max_sentences is not None and len(corpus_sentences) >= max_sentences:
                break

            if progress_every and (len(corpus_sentences) % progress_every == 0):
                print(f"  collected {len(corpus_sentences):,} sentences | elapsed={time.time()-t0:.1f}s")

        if max_sentences is not None and len(corpus_sentences) >= max_sentences:
            break

    print(f"Total sentences used for BM25: {len(corpus_sentences):,}")
    print("Tokenizing corpus for BM25...")
    tokenized_corpus = [bm25_tokenize(s) for s in corpus_sentences]

    print("Building BM25 index...")
    bm25 = BM25Okapi(tokenized_corpus)

    print(f"BM25 ready. elapsed={time.time()-t0:.1f}s")
    return bm25, corpus_sentences


def bm25_get_topk_with_timing_fast(
    bm25: BM25Okapi,
    corpus_sentences: List[str],
    query: str,
    k: int
) -> Tuple[List[str], float, List[float]]:
    """
    "Thời gian thật" cho từng top_k nhưng KHÔNG gọi BM25 nhiều lần.
    - 1 lần bm25.get_scores() cho toàn corpus (BM25 scoring)
    - đo mốc thời gian top_1..top_k bằng argpartition/argsort (nhẹ)

    Returns:
      - top_sents_k: list câu top-k (k = TOP_K)
      - scores_ms: thời gian tính scores cho toàn corpus (ms)
      - topk_ms_list: độ dài k, phần tử i là elapsed (ms) để có top_(i+1)
                      (scores_ms + thời gian chọn top_(i+1))
    """
    qtok = bm25_tokenize(query)

    t0 = time.time()
    scores = bm25.get_scores(qtok)  # numpy array length = len(corpus_sentences)
    scores_ms = (time.time() - t0) * 1000.0

    topk_ms_list: List[float] = []

    for kk in range(1, k + 1):
        t1 = time.time()
        idx_part = np.argpartition(scores, -kk)[-kk:]
        idx_sorted = idx_part[np.argsort(scores[idx_part])[::-1]]
        select_ms = (time.time() - t1) * 1000.0
        topk_ms_list.append(scores_ms + select_ms)

    idx_part = np.argpartition(scores, -k)[-k:]
    idx_sorted = idx_part[np.argsort(scores[idx_part])[::-1]]
    top_sents_k = [corpus_sentences[int(i)] for i in idx_sorted]

    return top_sents_k, scores_ms, topk_ms_list


class BioBERTYesNoMaybe:
    def __init__(self, model_path: str, device: Optional[str] = None, max_length: int = 256):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.max_length = max_length

        print("Loading tokenizer...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=True)

        print("Loading model...")
        self.model = AutoModelForSequenceClassification.from_pretrained(model_path).to(self.device)
        self.model.eval()

        self.id2label = {int(k): v for k, v in self.model.config.id2label.items()} if hasattr(self.model.config, "id2label") else {}

        print("Model loaded on", self.device)
        print("torch.cuda.is_available() =", torch.cuda.is_available())
        print("num_labels =", getattr(self.model.config, "num_labels", None))
        print("id2label   =", self.id2label)
        print("model device =", next(self.model.parameters()).device)

    def _map_label(self, pred_id: int) -> str:
        raw = (self.id2label.get(pred_id, str(pred_id)) or "").lower()
        if raw in {"yes", "no", "maybe"}:
            return raw

        manual = {"label_0": "no", "label_1": "yes", "label_2": "maybe"}
        if raw in manual:
            return manual[raw]

        return {0: "no", 1: "yes", 2: "maybe"}.get(pred_id, "maybe")

    @torch.inference_mode()
    def predict_batch(self, questions: List[str], contexts: List[str], batch_size: int = 64, log_every_batches: int = 2) -> List[str]:
        assert len(questions) == len(contexts), "questions and contexts must have same length"
        n = len(questions)
        out: List[str] = []

        device = next(self.model.parameters()).device
        t0 = time.time()
        num_batches = (n + batch_size - 1) // batch_size

        for b, i in enumerate(range(0, n, batch_size), start=1):
            q = questions[i:i + batch_size]
            c = contexts[i:i + batch_size]

            if log_every_batches and (b == 1 or b % log_every_batches == 0 or b == num_batches):
                elapsed = time.time() - t0
                done = min(i, n)
                eta = (elapsed / max(b, 1)) * (num_batches - b)
                print(f"infer batch {b}/{num_batches} | samples {done}/{n} | elapsed={elapsed:.1f}s | ETA={eta/60:.1f} min")

            enc = self.tokenizer(
                q, c,
                truncation=True,
                max_length=self.max_length,
                padding=True,
                return_tensors="pt",
            )
            enc = {k: v.to(device) for k, v in enc.items()}

            logits = self.model(**enc).logits
            pred_ids = torch.argmax(logits, dim=-1).tolist()
            out.extend([self._map_label(pid) for pid in pred_ids])

        return out


def run_pipeline():
    t0 = time.time()

    print("Loading PQA-L...")
    with open(PQA_L_TEST_PATH, "r", encoding="utf-8") as f:
        pqa_l: Dict[str, Any] = json.load(f)
    print("PQA-L samples (raw):", len(pqa_l), "| elapsed=", f"{time.time()-t0:.1f}s")

    if LIMIT_PQA_L_SAMPLES is not None:
        pqa_l = dict(list(pqa_l.items())[:LIMIT_PQA_L_SAMPLES])
        print("PQA-L samples (limited):", len(pqa_l), "| limit=", LIMIT_PQA_L_SAMPLES)

    print("Loading PQA-U...")
    with open(PQA_U_PATH, "r", encoding="utf-8") as f:
        pqa_u: Dict[str, Any] = json.load(f)
    print("PQA-U docs:", len(pqa_u), "| elapsed=", f"{time.time()-t0:.1f}s")

    bm25, corpus_sentences = build_bm25_from_pqa_u_sentence_level(
        pqa_u,
        max_sentences=MAX_BM25_SENTENCES,
        progress_every=PROGRESS_EVERY
    )

    clf = BioBERTYesNoMaybe(BIOBERT_PATH, max_length=MAX_LENGTH)

    items = list(pqa_l.items())
    pubids = [k for k, _ in items]
    questions = [normalize_ws(v.get("QUESTION", "")) for _, v in items]
    context_flats = [flatten_contexts(v.get("CONTEXTS", [])) for _, v in items]

    print(f"Running batch inference (batch_size={BATCH_SIZE}, max_length={MAX_LENGTH})...")
    t_pred = time.time()
    predicted_labels = clf.predict_batch(questions, context_flats, batch_size=BATCH_SIZE, log_every_batches=2)
    infer_elapsed = time.time() - t_pred
    infer_avg_ms = 1000.0 * infer_elapsed / max(len(items), 1)
    print("Inference done. elapsed=", f"{infer_elapsed:.1f}s", "| avg=", f"{infer_avg_ms:.3f} ms/sample")

    time_cols = ["bm25_ms", "bm25_scores_ms"] + [f"bm25_top{i}_ms" for i in range(1, TOP_K + 1)]
    fieldnames = ["pubid", "question", "context_flat", "predicted_label"] + [f"top_{i}" for i in range(1, TOP_K + 1)] + time_cols

    print("Writing CSV to:", OUTPUT_CSV_PATH)

    bm25_sum_ms = 0.0
    bm25_scores_sum_ms = 0.0
    bm25_top_sum_ms = [0.0] * TOP_K

    with open(OUTPUT_CSV_PATH, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=fieldnames,
            delimiter=",",
            quoting=csv.QUOTE_ALL,
            escapechar="\\"
        )
        writer.writeheader()

        t_write = time.time()
        for idx in range(len(items)):
            pubid = pubids[idx]
            question = questions[idx]
            context_flat = context_flats[idx]
            predicted_label = predicted_labels[idx]

            if BM25_QUERY_MODE == "question":
                query = question
            elif BM25_QUERY_MODE == "question_plus_context":
                query = normalize_ws(question + " " + context_flat)
            else:
                raise ValueError("BM25_QUERY_MODE must be 'question' or 'question_plus_context'")

            top_sents, scores_ms, topk_ms_list = bm25_get_topk_with_timing_fast(
                bm25=bm25,
                corpus_sentences=corpus_sentences,
                query=query,
                k=TOP_K
            )

            bm25_ms = topk_ms_list[-1] 

            row = {
                "pubid": pubid,
                "question": question,
                "context_flat": context_flat,
                "predicted_label": predicted_label,
                "bm25_ms": f"{bm25_ms:.3f}",
                "bm25_scores_ms": f"{scores_ms:.3f}",
            }

            for i in range(TOP_K):
                row[f"bm25_top{i+1}_ms"] = f"{topk_ms_list[i]:.3f}"

            for i in range(TOP_K):
                row[f"top_{i+1}"] = json.dumps(top_sents[: i+1], ensure_ascii=False)

            writer.writerow(row)

            bm25_sum_ms += bm25_ms
            bm25_scores_sum_ms += scores_ms
            for i in range(TOP_K):
                bm25_top_sum_ms[i] += topk_ms_list[i]

            if WRITE_LOG_EVERY and (idx + 1) % WRITE_LOG_EVERY == 0:
                elapsed = time.time() - t_write
                rate = elapsed / (idx + 1)
                eta = rate * (len(items) - (idx + 1))
                print(f"  wrote {idx+1}/{len(items)} rows | {rate:.3f}s/row | ETA={eta/60:.1f} min")

            if TIMING_LOG_EVERY and (idx + 1) % TIMING_LOG_EVERY == 0:
                n_done = idx + 1
                print(
                    f"[timing] after {n_done} samples | "
                    f"avg bm25_ms(k={TOP_K})={bm25_sum_ms/max(n_done,1):.3f} | "
                    f"avg bm25_scores_ms={bm25_scores_sum_ms/max(n_done,1):.3f} | "
                    f"avg infer_ms={infer_avg_ms:.3f}"
                )

    n = max(len(items), 1)
    print(f"AVG BM25 (k={TOP_K}): {bm25_sum_ms / n:.3f} ms/sample")
    print(f"AVG BM25 scores only: {bm25_scores_sum_ms / n:.3f} ms/sample")
    for i in range(TOP_K):
        print(f"AVG BM25 top_{i+1}: {bm25_top_sum_ms[i] / n:.3f} ms/sample")
    print(f"AVG BioBERT inference: {infer_avg_ms:.3f} ms/sample")

    print("DONE. Total elapsed:", f"{time.time()-t0:.1f}s")


run_pipeline()

# LLAMA 2_7B_GENERATION

In [ ]:
!pip -q install -U pip
!pip -q install --force-reinstall "numpy==2.0.2" "pandas==2.2.2"
!pip -q install --force-reinstall \
  "transformers==4.41.2" \
  "tokenizers==0.19.1" \
  "accelerate==0.30.1" \
  "huggingface-hub==0.23.4" \
  "safetensors>=0.4.3" \
  "bert-score==0.3.13"

In [ ]:
!pip -q install -U llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121

In [ ]:
import os, json, ast, re, time
import numpy as np
import pandas as pd
from tqdm import tqdm
import torch

print("torch:", torch.__version__, "cuda:", torch.cuda.is_available())
assert torch.cuda.is_available(), "CUDA is not available. Please enable GPU runtime in Colab."

from bert_score import score as bert_score
from llama_cpp import Llama

print("Imports OK: bert_score + llama_cpp")

BM25_PATH = "/content/drive/MyDrive/lap/lap_lap/pqaL_test_biobert_bm25_top5.csv"
PQAL_PATH = "/content/drive/MyDrive/BioBERT_PubMedQA_Checkpoints/ori_pqal.json"

MODEL_PATH = "/content/drive/MyDrive/BioBERT_PubMedQA_Checkpoints/llama-2-7b-chat.Q4_0.gguf"

N_SAMPLES = 500
SEED = 42

OUT_JSON = "/content/drive/MyDrive/lap/lap_lap/pqal_eval_500.json"
OUT_METRICS_CSV = "/content/drive/MyDrive/lap/lap_lap/pqal_metrics_by_k.csv"
OUT_TABLE_MD = "/content/drive/MyDrive/lap/lap_lap/pqal_table.md"

BERTSCORE_MODEL = "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext"

N_CTX = 2048
MAX_TOKENS = 120
TEMPERATURE = 0.2
TOP_P = 0.9

N_GPU_LAYERS = 35

MAX_DOC_CHARS = 900
MAX_EVIDENCE_DOCS_CAP = 5

# [TIMING] evaluate k range
K_MIN = 1
K_MAX = 5

np.random.seed(SEED)

def find_gguf_under_mydrive(max_results=20):
    root = "/content/drive/MyDrive"
    hits = []
    for dirpath, _, filenames in os.walk(root):
        for fn in filenames:
            if fn.lower().endswith(".gguf"):
                hits.append(os.path.join(dirpath, fn))
                if len(hits) >= max_results:
                    return hits
    return hits

if MODEL_PATH is None:
    ggufs = find_gguf_under_mydrive(max_results=50)
    if not ggufs:
        raise FileNotFoundError("No .gguf found under /content/drive/MyDrive. Upload your .gguf to Drive first.")
    print("Found GGUF files (first 10):")
    for p in ggufs[:10]:
        print(" -", p)
    MODEL_PATH = ggufs[0]
    print("\nUsing MODEL_PATH =", MODEL_PATH)

assert os.path.exists(MODEL_PATH), f"Model path does not exist: {MODEL_PATH}"

def load_bm25_file(path: str) -> pd.DataFrame:
    try:
        df = pd.read_csv(path, sep="\t", dtype=str)
        if "pubid" in df.columns and "question" in df.columns:
            return df
    except Exception:
        pass
    return pd.read_csv(path, sep=",", dtype=str)

def parse_list_cell(x):
    if x is None:
        return []
    s = str(x).strip()
    if not s or s.lower() == "nan":
        return []
    try:
        v = json.loads(s)
        return v if isinstance(v, list) else [str(v)]
    except Exception:
        pass
    try:
        v = ast.literal_eval(s)
        return v if isinstance(v, list) else [str(v)]
    except Exception:
        return []

def build_evidence_from_topk_row(row: pd.Series, k: int):
    docs = parse_list_cell(row[f"top_{k}"])
    docs = [str(d).strip() for d in docs if str(d).strip()]
    docs = docs[:min(k, MAX_EVIDENCE_DOCS_CAP, len(docs))]
    return docs

def first_sentence(text: str) -> str:
    text = (text or "").strip()
    if not text:
        return text
    parts = re.split(r'(?<=[.!?])\s+', text)
    return parts[0].strip() if parts else text

assert os.path.exists(PQAL_PATH), f"PQAL not found: {PQAL_PATH}"
with open(PQAL_PATH, "r", encoding="utf-8") as f:
    pqal = json.load(f)

pqal_rows = []
for pubid, obj in pqal.items():
    pqal_rows.append({
        "pubid": str(pubid),
        "QUESTION": obj.get("QUESTION", ""),
        "LONG_ANSWER": obj.get("LONG_ANSWER", ""),
        "final_decision": obj.get("final_decision", ""),
    })
pqal_df = pd.DataFrame(pqal_rows)
print("PQAL loaded:", pqal_df.shape)
display(pqal_df.head(2))

assert os.path.exists(BM25_PATH), f"BM25 not found: {BM25_PATH}"
bm25_df = load_bm25_file(BM25_PATH)
print("BM25 loaded:", bm25_df.shape)
print("BM25 columns:", list(bm25_df.columns))

needed = ["pubid", "question"] + [f"top_{k}" for k in range(K_MIN, K_MAX + 1)]
missing = [c for c in needed if c not in bm25_df.columns]
if missing:
    raise ValueError(f"BM25 file missing columns: {missing}")

bm25_df["pubid"] = bm25_df["pubid"].astype(str)

df = bm25_df.merge(pqal_df[["pubid", "LONG_ANSWER", "final_decision"]], on="pubid", how="inner")
df["question"] = df["question"].fillna("").astype(str)
df["LONG_ANSWER"] = df["LONG_ANSWER"].fillna("").astype(str)

df = df[(df["question"].str.len() > 10) & (df["LONG_ANSWER"].str.len() > 20)].copy()
print("Joined usable rows:", df.shape)

df_sample = df.sample(n=min(N_SAMPLES, len(df)), random_state=SEED).reset_index(drop=True)
print("Using sample:", df_sample.shape)

print("\nLoading Llama (GPU offload)...")
llm = Llama(
    model_path=MODEL_PATH,
    n_ctx=N_CTX,
    n_threads=os.cpu_count(),
    n_gpu_layers=N_GPU_LAYERS,
    verbose=True,
)
print("Llama loaded.\n")

def llama_generate_final(question: str, evidence_docs: list[str]) -> str:
    evidence_block = "\n".join([f"{i+1}. {doc[:MAX_DOC_CHARS]}" for i, doc in enumerate(evidence_docs)])

    prompt = f"""[INST] You are a medical QA assistant.
Use ONLY the evidence. If evidence is insufficient, say "Insufficient evidence."

Question: {question}

Evidence:
{evidence_block}

Write ONLY ONE final concluding sentence answering the question (no extra explanation). [/INST]
"""
    out = llm(
        prompt,
        max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        stop=["</s>", "[INST]", "[/INST]"],
        echo=False
    )
    return first_sentence(out["choices"][0]["text"].strip())

print("Quick test generation (watch VRAM in another cell with `!nvidia-smi -l 1`):")
test = llama_generate_final("Is aspirin used to reduce fever?", ["Aspirin is used to reduce pain and fever."])
print("Test output:", test)

results = []
gen_errors = 0

gen_time_sum_ms = {k: 0.0 for k in range(K_MIN, K_MAX + 1)}
gen_time_n_ok = {k: 0 for k in range(K_MIN, K_MAX + 1)}
gen_time_n_all = {k: 0 for k in range(K_MIN, K_MAX + 1)}  # including errors

t_gen_all0 = time.time()

for _, row in tqdm(df_sample.iterrows(), total=len(df_sample)):
    pubid = row["pubid"]
    q = row["question"]
    gt_long = row["LONG_ANSWER"]
    gt_label = row.get("final_decision", "")

    item = {
        "pubid": pubid,
        "question": q,
        "ground_truth_long": gt_long,
        "ground_truth_label": gt_label,
        "per_k": {}
    }

    for k in range(K_MIN, K_MAX + 1):
        gen_time_n_all[k] += 1
        evidence_docs = build_evidence_from_topk_row(row, k)

        try:
            t0 = time.time()
            gen = llama_generate_final(q, evidence_docs)
            gen_ms = (time.time() - t0) * 1000.0

            gen_time_sum_ms[k] += gen_ms
            gen_time_n_ok[k] += 1

            item["per_k"][str(k)] = {
                "k": k,
                "generated": gen,
                "evidence_docs": evidence_docs,
                "gen_ms": float(gen_ms),  
            }
        except Exception as e:
            gen_errors += 1
            item["per_k"][str(k)] = {
                "k": k,
                "generated": "",
                "evidence_docs": evidence_docs,
                "gen_ms": None,            
                "error": str(e)
            }

    results.append(item)

gen_elapsed_s = time.time() - t_gen_all0
print("Generation done. items:", len(results), "errors:", gen_errors, "| elapsed:", f"{gen_elapsed_s:.1f}s")

print("\nAVG generation time per k:")
for k in range(K_MIN, K_MAX + 1):
    n_ok = gen_time_n_ok[k]
    avg_ms = (gen_time_sum_ms[k] / n_ok) if n_ok > 0 else float("nan")
    print(f"  k={k}: n_ok={n_ok}/{gen_time_n_all[k]} | avg_gen_ms={avg_ms:.2f}")

with open(OUT_JSON, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print("Saved:", OUT_JSON)

metrics = []

t_bs_all0 = time.time()

for k in range(K_MIN, K_MAX + 1):
    cands, refs = [], []
    for item in results:
        gen = item["per_k"].get(str(k), {}).get("generated", "")
        ref = item.get("ground_truth_long", "")
        if gen and ref:
            cands.append(gen)
            refs.append(ref)

    n = len(cands)
    if n == 0:
        metrics.append({
            "k": k,
            "n": 0,
            "bertscore_P_mean": np.nan,
            "bertscore_R_mean": np.nan,
            "bertscore_F1_mean": np.nan,
            "bertscore_F1_std": np.nan,
            # [TIMING]
            "avg_gen_ms": (gen_time_sum_ms[k] / gen_time_n_ok[k]) if gen_time_n_ok[k] > 0 else np.nan,
            "gen_n_ok": gen_time_n_ok[k],
            "bertscore_total_ms": np.nan,
            "bertscore_avg_ms_per_pair": np.nan,
        })
        continue

    t0 = time.time()
    P, R, F1 = bert_score(
        cands, refs,
        lang="en",
        model_type=BERTSCORE_MODEL,
        verbose=False,
        rescale_with_baseline=True,
        device="cuda"
    )
    bs_ms = (time.time() - t0) * 1000.0

    P = P.detach().cpu().numpy()
    R = R.detach().cpu().numpy()
    F1 = F1.detach().cpu().numpy()

    metrics.append({
        "k": k,
        "n": n,
        "bertscore_P_mean": float(P.mean()),
        "bertscore_R_mean": float(R.mean()),
        "bertscore_F1_mean": float(F1.mean()),
        "bertscore_F1_std": float(F1.std()),
        # [TIMING]
        "avg_gen_ms": float(gen_time_sum_ms[k] / gen_time_n_ok[k]) if gen_time_n_ok[k] > 0 else np.nan,
        "gen_n_ok": int(gen_time_n_ok[k]),
        "bertscore_total_ms": float(bs_ms),
        "bertscore_avg_ms_per_pair": float(bs_ms / max(n, 1)),
    })

bs_elapsed_s = time.time() - t_bs_all0
print("\nBERTScore done. elapsed:", f"{bs_elapsed_s:.1f}s")

metrics_df = pd.DataFrame(metrics).sort_values("k")
metrics_df.to_csv(OUT_METRICS_CSV, index=False)
print("Saved:", OUT_METRICS_CSV)
display(metrics_df)

md = []
md.append("| k | n | F1 mean | F1 std | avg_gen_ms | bertscore_total_ms | bertscore_avg_ms/pair |")
md.append("|---:|---:|---:|---:|---:|---:|---:|")
for _, r in metrics_df.iterrows():
    k = int(r["k"])
    n = int(r["n"]) if not pd.isna(r["n"]) else 0
    f1m = r["bertscore_F1_mean"]
    f1s = r["bertscore_F1_std"]
    ag  = r["avg_gen_ms"]
    bt  = r["bertscore_total_ms"]
    bap = r["bertscore_avg_ms_per_pair"]

    md.append(
        f"| {k} | {n} | "
        f"{(f1m if not pd.isna(f1m) else float('nan')):.4f} | "
        f"{(f1s if not pd.isna(f1s) else float('nan')):.4f} | "
        f"{(ag  if not pd.isna(ag)  else float('nan')):.2f} | "
        f"{(bt  if not pd.isna(bt)  else float('nan')):.2f} | "
        f"{(bap if not pd.isna(bap) else float('nan')):.2f} |"
    )

with open(OUT_TABLE_MD, "w", encoding="utf-8") as f:
    f.write("\n".join(md))

print("Saved:", OUT_TABLE_MD)
print("\n".join(md))